<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Hunyuan3D-3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hunyuan3D 3.0 — Tencent's Cloud API Wrapper

A Colab-friendly wrapper around the [Tencent Cloud Hunyuan 3D Global API v3](https://www.tencentcloud.com/document/product/1665/119114), adapted from [exedesign/Hunyuan-3D-v3](https://github.com/exedesign/Hunyuan-3D-v3) (which was built for ComfyUI).

Tencent's flagship 3D generation model is not yet open-weights — but the **Global API** is available for anyone to call with a paid Tencent Cloud account. This notebook lets you generate `.glb` meshes from **text prompts or images** without any local GPU or weight download.

**Two modes:**

- **Text-to-3D** — describe the model in natural language, get a `.glb` back
- **Image-to-3D** — upload a reference image (PNG / JPG / WEBP), get a `.glb` back

**Configuration options:**

- `EnablePBR` — generate PBR (albedo / metallic / roughness) textures (default: off)
- `FaceCount` — 40,000 to 1,500,000 faces (default: 500,000)
- `GenerateType` — `Normal` (default), `LowPoly`, `Geometry`, or `Sketch`
- `PolygonType` — `triangle` (default) or `quadrilateral` (only used with `LowPoly`)

**Cost:** ~$0.10-0.60 per request, billed to your Tencent Cloud account. The notebook has no built-in cost controls — set a billing alert on the Tencent side. We recommend adding **$10-20** of credit before running a batch.

**Runtime:** any Colab runtime works (even CPU). The heavy work is on Tencent's side; we just poll a job-id.

**Setup:** you'll need a [Tencent Cloud international account](https://console.intl.cloud.tencent.com/), a SecretId + SecretKey, and the **Hunyuan 3D** service activated in the AI Services console. The notebook reads the credentials from Colab secrets.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the Tencent Cloud SDK (one small package, no model weights).
# @markdown Also sets up Colab secrets for the API credentials.

import os, sys, subprocess, time

print('[Runtime] Hunyuan3D 3.0 runs on Tencent\'s cloud — no GPU or weight download needed.')
t0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
               'tencentcloud-sdk-python-ai3d>=3.0.1490',
               'aiohttp',
               'Pillow',
               'gradio==5.49.1',
               'nest-asyncio',
               'tqdm'], check=True)
print(f'[Install] DONE in {time.time() - t0:.1f}s.')

# Confirm the SDK is importable.
try:
    from tencentcloud.common import credential
    from tencentcloud.common.common_client import CommonClient
    from tencentcloud.common.profile.client_profile import ClientProfile
    from tencentcloud.common.profile.http_profile import HttpProfile
    print('[Install] tencentcloud-sdk-python-ai3d imports OK.')
except Exception as e:
    raise SystemExit(f'Tencent Cloud SDK import failed: {e}')

# Read API credentials from Colab secrets.
try:
    from google.colab import userdata
    SECRET_ID = userdata.get('TENCENT_SECRET_ID')
    SECRET_KEY = userdata.get('TENCENT_SECRET_KEY')
    if not SECRET_ID or not SECRET_KEY:
        print()
        print('=' * 60)
        print('MISSING CREDENTIALS')
        print('=' * 60)
        print('Set the following in the Colab secrets panel (left sidebar → \U0001F511):')
        print('  - TENCENT_SECRET_ID       (e.g. AKIDxxxxxxxxxxxxxxxx)')
        print('  - TENCENT_SECRET_KEY      (e.g. xxxxxxxxxxxxxxxxxxxxxxxx)')
        print('Get them from: https://console.intl.cloud.tencent.com/cam/capi')
        print('Then re-run this cell.')
    else:
        os.environ['TENCENT_SECRET_ID'] = SECRET_ID
        os.environ['TENCENT_SECRET_KEY'] = SECRET_KEY
        print(f'[Credentials] Loaded TENCENT_SECRET_ID and TENCENT_SECRET_KEY from Colab secrets.')
except Exception as e:
    print(f'[Credentials] Could not read secrets: {e}')
    print('You can also set them in the UI when you run Step 4.')

print()
print('\u26a0\ufe0f  COST WARNING: Hunyuan 3D is a paid API. Each request costs \u2248$0.10-0.60.')
print('   Set a billing alert on your Tencent Cloud account before running batches.')

In [ ]:
# @title STEP 2 — Connectivity Check
# @markdown Verifies that the Tencent Cloud API is reachable and your credentials work.
# @markdown No weights to download — the model runs on Tencent\'s servers.

import os, sys, asyncio, traceback

def _check_reachable(secret_id=None, secret_key=None):
    """Try a tiny SubmitHunyuanTo3DProJob call to verify credentials.
    We submit a minimal text-to-3D job with a short prompt; we don't
    wait for it to finish, just check that the API accepted the request.
    """
    from tencentcloud.common import credential
    from tencentcloud.common.common_client import CommonClient
    from tencentcloud.common.profile.client_profile import ClientProfile
    from tencentcloud.common.profile.http_profile import HttpProfile
    try:
        cred = credential.Credential(secret_id or os.environ.get('TENCENT_SECRET_ID', ''),
                                    secret_key or os.environ.get('TENCENT_SECRET_KEY', ''))
        httpProfile = HttpProfile()
        httpProfile.endpoint = 'hunyuan.intl.tencentcloudapi.com'
        clientProfile = ClientProfile()
        clientProfile.httpProfile = httpProfile
        client = CommonClient('hunyuan', '2023-09-01', cred, 'ap-singapore', profile=clientProfile)
        # Submit a minimal job. The prompt is short so the cost is minimal if the call succeeds.
        resp = client.call_json('SubmitHunyuanTo3DProJob', {
            'Prompt': 'connectivity probe',
            'EnablePBR': False,
            'FaceCount': 40000,
            'GenerateType': 'LowPoly',
        })
        job_id = resp['Response'].get('JobId', '')
        return True, f'OK — job submitted, JobId={job_id[:16]}...'
    except Exception as e:
        msg = str(e)
        if 'ResourceInsufficient' in msg:
            return False, 'ResourceInsufficient — add credits to your account.'
        if 'AuthFailure' in msg:
            return False, 'AuthFailure — check your SecretId/SecretKey.'
        if 'InvalidParameter' in msg:
            return True, 'InvalidParameter — credentials OK, but the probe prompt was rejected (normal).'
        return False, f'{type(e).__name__}: {msg[:200]}'

if os.environ.get('TENCENT_SECRET_ID') and os.environ.get('TENCENT_SECRET_KEY'):
    ok, msg = _check_reachable()
    if ok:
        print(f'[Connectivity] {msg}')
    else:
        print(f'[Connectivity] \u274c {msg}')
        print('  → Check your credentials, billing, and that the Hunyuan 3D service is activated.')
else:
    print('[Connectivity] Skipped — no credentials in env. Set TENCENT_SECRET_ID and TENCENT_SECRET_KEY in Colab secrets and re-run.')
    print('  → You can also paste them into the UI in Step 4.')

# Create the output directory on Drive (only if Drive is mounted, so this works in any runtime).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Hunyuan3D-3')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'[Drive] Output cache: {DRIVE_OUT}')
    HAS_DRIVE = True
except Exception:
    DRIVE_OUT = pathlib.Path('/content/hunyuan3d3_out')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'[Drive] Not available, using local cache: {DRIVE_OUT}')
    HAS_DRIVE = False

print('\n[Setup] DONE. Run Step 3 next.')

In [ ]:
# @title STEP 3 — Engine: Hunyuan3Dv3Client (sync wrappers)
# @markdown Defines a thin client class with synchronous wrappers for the async API calls.
# @markdown The client is the only thing the UI needs.

import os, sys, time, asyncio, json, base64, io, traceback, uuid, threading, pathlib
from typing import Optional, Callable
import aiohttp, requests
from PIL import Image

# ---- async helpers (run in a dedicated thread so we can call from sync code) ----
_LOOP = None
_LOOP_THREAD = None
def _get_loop():
    global _LOOP, _LOOP_THREAD
    if _LOOP is not None and _LOOP.is_running():
        return _LOOP
    _LOOP = asyncio.new_event_loop()
    def _runner():
        asyncio.set_event_loop(_LOOP)
        _LOOP.run_forever()
    _LOOP_THREAD = threading.Thread(target=_runner, daemon=True)
    _LOOP_THREAD.start()
    return _LOOP

def _run(coro):
    """Run an async coroutine in the background loop and return its result."""
    loop = _get_loop()
    fut = asyncio.run_coroutine_threadsafe(coro, loop)
    return fut.result()

# ---- client ----
class Hunyuan3Dv3Client:
    """Synchronous wrapper around the Tencent Cloud Hunyuan 3D Global API v3."""

    ENDPOINT = 'hunyuan.intl.tencentcloudapi.com'
    SERVICE = 'hunyuan'
    VERSION = '2023-09-01'
    REGION = 'ap-singapore'
    SUBMIT_METHOD = 'SubmitHunyuanTo3DProJob'
    QUERY_METHOD = 'QueryHunyuanTo3DProJob'

    def __init__(self, secret_id: str, secret_key: str, region: str = 'ap-singapore'):
        self.secret_id = secret_id
        self.secret_key = secret_key
        self.region = region
        self._client = None
        if not secret_id or not secret_key:
            print('[Client] WARNING: empty credentials. Set them in the UI or in env.')
            return
        from tencentcloud.common import credential
        from tencentcloud.common.common_client import CommonClient
        from tencentcloud.common.profile.client_profile import ClientProfile
        from tencentcloud.common.profile.http_profile import HttpProfile
        cred = credential.Credential(secret_id, secret_key)
        httpProfile = HttpProfile()
        httpProfile.endpoint = self.ENDPOINT
        clientProfile = ClientProfile()
        clientProfile.httpProfile = httpProfile
        self._client = CommonClient(self.SERVICE, self.VERSION, cred, region, profile=clientProfile)
        print(f'[Client] Initialized for region {region}')

    @property
    def is_ready(self) -> bool:
        return self._client is not None

    # -------- public API --------
    def text_to_3d(self, prompt: str, enable_pbr: bool = False, face_count: int = 500000,
                   generate_type: str = 'Normal', polygon_type: str = 'triangle') -> str:
        if not self.is_ready: raise RuntimeError('client not ready — set credentials first')
        params = {
            'Prompt': prompt,
            'EnablePBR': bool(enable_pbr),
            'FaceCount': int(face_count),
            'GenerateType': generate_type,
        }
        if generate_type == 'LowPoly':
            params['PolygonType'] = polygon_type
        resp = self._client.call_json(self.SUBMIT_METHOD, params)
        return resp['Response']['JobId']

    def image_to_3d(self, image_input, enable_pbr: bool = False, face_count: int = 500000,
                    generate_type: str = 'Normal', polygon_type: str = 'triangle') -> str:
        if not self.is_ready: raise RuntimeError('client not ready — set credentials first')
        # accept PIL.Image or file path
        if isinstance(image_input, Image.Image):
            buf = io.BytesIO()
            image_input.convert('RGBA').save(buf, format='PNG')
            image_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
        else:
            with open(image_input, 'rb') as f:
                image_b64 = base64.b64encode(f.read()).decode('utf-8')
        params = {
            'ImageBase64': image_b64,
            'EnablePBR': bool(enable_pbr),
            'FaceCount': int(face_count),
            'GenerateType': generate_type,
        }
        if generate_type == 'LowPoly':
            params['PolygonType'] = polygon_type
        resp = self._client.call_json(self.SUBMIT_METHOD, params)
        return resp['Response']['JobId']

    def query(self, job_id: str) -> dict:
        if not self.is_ready: raise RuntimeError('client not ready')
        resp = self._client.call_json(self.QUERY_METHOD, {'JobId': job_id})
        r = resp['Response']
        result = {
            'status': r.get('Status', ''),
            'error_code': r.get('ErrorCode', ''),
            'error_message': r.get('ErrorMessage', ''),
            'glb_url': None,
            'preview_url': None,
        }
        for f in r.get('ResultFile3Ds', []):
            if f.get('Type', '').upper() == 'GLB':
                result['glb_url'] = f.get('Url')
                result['preview_url'] = f.get('PreviewImageUrl')
        return result

    def wait_for_completion(self, job_id: str, max_wait: int = 600, poll: int = 5,
                             on_progress: Optional[Callable] = None) -> dict:
        t0 = time.time()
        while time.time() - t0 < max_wait:
            r = self.query(job_id)
            elapsed = int(time.time() - t0)
            status = r['status']
            if status == 'DONE':
                if on_progress: on_progress(100, 'Done.')
                return r
            if status == 'FAIL':
                raise RuntimeError(f"Job failed: {r['error_code']} — {r['error_message']}")
            if on_progress:
                pct = min(95, 5 + (elapsed / 180.0) * 90)
                on_progress(pct, f'Status={status}, {elapsed}s elapsed')
            time.sleep(poll)
        raise TimeoutError(f'Job {job_id} did not complete within {max_wait}s')

    def download_glb(self, url: str, out_path: str) -> str:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        pathlib.Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, 'wb') as f:
            f.write(r.content)
        return out_path

# Singleton, lazily initialized with the credentials from the UI.
CLIENT = Hunyuan3Dv3Client(
    secret_id=os.environ.get('TENCENT_SECRET_ID', ''),
    secret_key=os.environ.get('TENCENT_SECRET_KEY', ''),
)
print('[Core] Engine ready. Use Step 4 to launch the UI.')

In [ ]:

# @title STEP 4 — Launch Gradio UI
# @markdown Opens a Gradio app for text-to-3D, image-to-3D, and status.

import os, sys, time, json, uuid, traceback, pathlib
import torch, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink

OUT_DIR = DRIVE_OUT if HAS_DRIVE else pathlib.Path('/content/hunyuan3d3_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ---- Cost estimator (heuristic; Tencent doesn't publish exact per-call rates) ----
# Base costs in USD per call, by generate_type
_BASE_COST = {
    'LowPoly':  0.05,  # 40K-face, no PBR baseline
    'Normal':   0.20,  # 500K-face, no PBR baseline
    'Geometry': 0.10,  # shape only, no texture
    'Sketch':   0.25,  # hand-drawn reference, slightly higher
}
# Multipliers
_PBR_MULT         = 1.50   # PBR adds ~50% (extra albedo+mr+preview images)
_FACE_COST_SCALE  = 0.10   # log-scaled face-count adder


def _estimate_cost(enable_pbr, face_count, generate_type):
    """Heuristic cost estimate in USD. Returns (cost_str, cost_float)."""
    base = _BASE_COST.get(generate_type, 0.20)
    # log-scaled face-count adder (40K -> +0, 500K -> +0.10, 1.5M -> +0.20)
    import math
    face_add = _FACE_COST_SCALE * max(0.0, math.log10(max(face_count, 40000) / 40000.0))
    cost = (base + face_add) * (_PBR_MULT if enable_pbr else 1.0)
    return f'~${cost:.2f}', cost


# Session-level counter (lives in module globals so it survives across Gradio calls)
_SESSION = {'count': 0, 'spent_usd': 0.0}


def _session_status():
    return (f'Jobs run this session: {_SESSION["count"]}\n'
            f'Estimated spend (heuristic): ${_SESSION["spent_usd"]:.2f}\n'
            f'Output cache: {OUT_DIR}\n'
            f'Credentials loaded: {CLIENT.is_ready}')


def _record_job(cost_float):
    _SESSION['count'] += 1
    _SESSION['spent_usd'] += cost_float



def _new_out(suffix: str = '.glb') -> pathlib.Path:
    p = OUT_DIR / f'{int(time.time())}_{uuid.uuid4().hex[:6]}{suffix}'
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def _err(msg: str):
    return None, 'ERROR: ' + msg


def do_text_to_3d(prompt, enable_pbr, face_count, generate_type, polygon_type, progress=gr.Progress()):
    if not prompt or not prompt.strip():
        return _err('Type a prompt first.')
    if not CLIENT.is_ready:
        return _err('Client not ready — set TENCENT_SECRET_ID and TENCENT_SECRET_KEY in Colab secrets, then re-run Step 1.')
    try:
        progress(0.05, desc='Submitting job to Tencent Cloud ...')
        job_id = CLIENT.text_to_3d(prompt.strip(), enable_pbr, face_count, generate_type, polygon_type)
        progress(0.10, desc='Job ' + job_id[:12] + ' submitted, waiting ...')

        def _cb(pct, msg):
            progress(min(0.95, pct / 100.0), desc=msg)

        result = CLIENT.wait_for_completion(job_id, max_wait=600, poll=5, on_progress=_cb)
        if not result.get('glb_url'):
            return _err('No GLB in result: ' + json.dumps(result))
        progress(0.97, desc='Downloading .glb ...')
        out = _new_out('.glb')
        CLIENT.download_glb(result['glb_url'], str(out))
        progress(1.0, desc='Done.')
        # record the cost estimate (heuristic, may differ from actual billing)
        _cost_str, _cost_val = _estimate_cost(enable_pbr, face_count, generate_type)
        _record_job(_cost_val)
        return str(out), 'Generated in ' + str(int(time.time())) + 's. Cost: ' + _cost_str + '. Session total: $' + f"{_SESSION['spent_usd']:.2f}" + '. JobId=' + job_id
    except Exception as e:
        traceback.print_exc()
        return _err(str(e))


def do_image_to_3d(image, enable_pbr, face_count, generate_type, polygon_type, progress=gr.Progress()):
    if image is None:
        return _err('Upload an image first.')
    if not CLIENT.is_ready:
        return _err('Client not ready.')
    try:
        progress(0.05, desc='Encoding image to base64 ...')
        if not isinstance(image, Image.Image):
            image = Image.open(image)
        progress(0.10, desc='Submitting image-to-3D job ...')
        job_id = CLIENT.image_to_3d(image, enable_pbr, face_count, generate_type, polygon_type)
        progress(0.15, desc='Job ' + job_id[:12] + ' submitted, waiting ...')

        def _cb(pct, msg):
            progress(min(0.95, pct / 100.0), desc=msg)

        result = CLIENT.wait_for_completion(job_id, max_wait=600, poll=5, on_progress=_cb)
        if not result.get('glb_url'):
            return _err('No GLB in result: ' + json.dumps(result))
        progress(0.97, desc='Downloading .glb ...')
        out = _new_out('.glb')
        CLIENT.download_glb(result['glb_url'], str(out))
        progress(1.0, desc='Done.')
        _cost_str, _cost_val = _estimate_cost(enable_pbr, face_count, generate_type)
        _record_job(_cost_val)
        return str(out), 'Generated. Cost: ' + _cost_str + '. Session total: $' + f"{_SESSION['spent_usd']:.2f}" + '. JobId=' + job_id
    except Exception as e:
        traceback.print_exc()
        return _err(str(e))


def vram_status():
    # No GPU usage, but show a status line.
    return ('No GPU used (Hunyuan 3D runs on Tencent Cloud).\\n'
            'Output cache: ' + str(OUT_DIR) + '\\n'
            'Credentials loaded: ' + str(CLIENT.is_ready))


EXAMPLE_PROMPTS = [
    ['A wooden treasure chest with iron bands, fantasy RPG style, 500k faces.'],
    ['A small red ceramic teapot, soft studio lighting, photorealistic.'],
    ['A cartoon dragon character, friendly, low-poly game-ready.'],
    ['A futuristic sci-fi helmet, scratched metal, PBR materials.'],
    ['A bonsai tree in a clay pot, traditional Japanese aesthetic.'],
    ['A vintage pocket watch, brass gears, steampunk.'],
]


with gr.Blocks(title='Hunyuan3D 3.0 Colab', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Hunyuan3D 3.0 — Tencent Cloud API\nTencent\'s flagship 3D model, no GPU required. ~$0.10-0.60 per request.')

    gr.HTML('<div style="background: #fef3c7; color: #92400e; padding: 8px 12px; border-radius: 6px; margin: 6px 0;">'
            '<b>Cost warning:</b> This is a paid API. Each call bills your Tencent Cloud account. '
            'Set a billing alert before running batches. Generation typically takes 1-3 minutes per request.'
            '</div>')

    if not CLIENT.is_ready:
        gr.HTML('<div style="background: #fee2e2; color: #991b1b; padding: 8px 12px; border-radius: 6px; margin: 6px 0;">'
                '<b>Credentials not found.</b> Add <code>TENCENT_SECRET_ID</code> and <code>TENCENT_SECRET_KEY</code> '
                'to Colab secrets (left sidebar → \\U0001F511) and re-run Step 1.'
                '</div>')

    with gr.Tabs():
        # -- Text-to-3D --
        with gr.Tab('Text-to-3D'):
            with gr.Row():
                with gr.Column():
                    prompt_in = gr.Textbox(
                        label='Prompt',
                        value='A small wooden treasure chest with iron bands and a sturdy lock, fantasy RPG style.',
                        lines=4,
                        info='Describe the object in detail: material, style, lighting, mood. The model follows detailed instructions well.'
                    )
                    enable_pbr_t = gr.Checkbox(value=False, label='PBR materials',
                                              info='Generate albedo + metallic + roughness maps. Adds ~50% to cost and latency.')
                    with gr.Accordion('Advanced options', open=False):
                        face_count_t = gr.Slider(40000, 1500000, value=500000, step=10000, label='Face count',
                                                  info='40K = low-poly (game-ready, fast), 500K = balanced (default), 1.5M = hero asset (slow, large file).')
                        gen_type_t = gr.Radio(['Normal', 'LowPoly', 'Geometry', 'Sketch'], value='Normal', label='Generate type',
                                              info='Normal = full quality. LowPoly = low-triangle, faster. Geometry = shape only (no texture). Sketch = best for hand-drawn references.')
                        poly_type_t = gr.Radio(['triangle', 'quadrilateral'], value='triangle', label='Polygon type (LowPoly only)',
                                               info='Only used when Generate type = LowPoly. Triangles are universal; quads are cleaner for retopology.')

                    def _update_cost_t(pbr, faces, gtype, ptype):
                        s, v = _estimate_cost(pbr, faces, gtype)
                        return s, v

                    for inp in (enable_pbr_t, face_count_t, gen_type_t, poly_type_t):
                        inp.change(_update_cost_t, inputs=[enable_pbr_t, face_count_t, gen_type_t, poly_type_t], outputs=[cost_t, cost_t_value])
                    # Initial value
                    cost_t.value, cost_t_value.value = _update_cost_t(False, 500000, 'Normal', 'triangle')
                    btn_t = gr.Button('Generate', variant='primary')
                    cost_t = gr.Markdown('', elem_id='cost-t')
                    cost_t_value = gr.State(0.0)
                with gr.Column():
                    file_out_t = gr.File(label='Download .glb')
                    status_t = gr.Markdown('')
            btn_t.click(do_text_to_3d, inputs=[prompt_in, enable_pbr_t, face_count_t, gen_type_t, poly_type_t],
                        outputs=[file_out_t, status_t])            .then(lambda: f'Session: {_SESSION["count"]} jobs, ~${_SESSION["spent_usd"]:.2f} (heuristic)', outputs=[cost_t])

        # -- Image-to-3D --
        with gr.Tab('Image-to-3D'):
            gr.Markdown('Upload a single image of an isolated object (clean background helps). PNG / JPG / WEBP.')
            with gr.Row():
                with gr.Column():
                    img_in = gr.Image(label='Reference image', type='pil', image_mode='RGBA', height=320,)
                    enable_pbr_i = gr.Checkbox(value=False, label='PBR materials',
                                                info='Generate albedo + metallic + roughness maps.')
                    with gr.Accordion('Advanced options', open=False):
                        face_count_i = gr.Slider(40000, 1500000, value=500000, step=10000, label='Face count')
                        gen_type_i = gr.Radio(['Normal', 'LowPoly', 'Geometry', 'Sketch'], value='Normal', label='Generate type')
                        poly_type_i = gr.Radio(['triangle', 'quadrilateral'], value='triangle', label='Polygon type (LowPoly only)')

                    def _update_cost_i(pbr, faces, gtype, ptype):
                        s, v = _estimate_cost(pbr, faces, gtype)
                        return s, v

                    for inp in (enable_pbr_i, face_count_i, gen_type_i, poly_type_i):
                        inp.change(_update_cost_i, inputs=[enable_pbr_i, face_count_i, gen_type_i, poly_type_i], outputs=[cost_i, cost_i_value])
                    cost_i.value, cost_i_value.value = _update_cost_i(False, 500000, 'Normal', 'triangle')
                    btn_i = gr.Button('Generate', variant='primary')
                    cost_i = gr.Markdown('', elem_id='cost-i')
                    cost_i_value = gr.State(0.0)
                with gr.Column():
                    file_out_i = gr.File(label='Download .glb')
                    status_i = gr.Markdown('')
            btn_i.click(do_image_to_3d, inputs=[img_in, enable_pbr_i, face_count_i, gen_type_i, poly_type_i],
                        outputs=[file_out_i, status_i])            .then(lambda: f'Session: {_SESSION["count"]} jobs, ~${_SESSION["spent_usd"]:.2f} (heuristic)', outputs=[cost_i])

        # -- VRAM (no GPU, but show output dir + creds) --
        with gr.Tab('Status'):
            vram_text = gr.Textbox(label='Status', value=vram_status, interactive=False, lines=6)
            gr.Button('Refresh status').click(vram_status, outputs=vram_text)

    gr.Examples(label='Quick starts (Text-to-3D)', examples=EXAMPLE_PROMPTS, inputs=[prompt_in], examples_per_page=6)

clear_output()
print('[UI] Launching ...')
demo.queue(max_size=4, default_concurrency_limit=2)
demo.load(lambda: 'Hunyuan3D 3.0 is ready. Each request bills ~$0.10-0.60 to your Tencent Cloud account.', outputs=[status_t])
demo.launch(share=True, show_error=True, height=720, prevent_thread_lock=False)
print('\\n[UI] Public URL above. Open it in a new tab.')

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings a tiny URL every 60 s to keep the Colab tab active.

import time, requests, datetime, threading, IPython
_stop = threading.Event()
def _pinger():
    while not _stop.is_set():
        try:
            requests.get('https://huggingface.co/api/models/tencent/Hunyuan3D-2.1', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} OK. Stop with `_stop.set()`.')
        except Exception as e:
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} WARN: {e}')
        _stop.wait(60)
_t = threading.Thread(target=_pinger, daemon=True)
_t.start()
print('[Keep-Alive] Started.')

In [ ]:
# @title STEP 6 — Quick Test (single API call, no UI)
# @markdown Submits one text-to-3D job with a low-poly short prompt and waits for it to finish.
# @markdown Cost: ~$0.10 (low-poly, 40k faces). Use this to confirm your credentials work.

import os, time, traceback, pathlib
from IPython.display import display, FileLink

QUICK_PROMPT = 'a small green gem, low-poly'  # @param {type:"string"}
QUICK_FACES = 40000  # @param {type:"integer"}
QUICK_TYPE = 'LowPoly'  # @param ["Normal", "LowPoly", "Geometry", "Sketch"]
QUICK_PBR = False  # @param {type:"boolean"}

if not CLIENT.is_ready:
    raise SystemExit('Client not ready — set TENCENT_SECRET_ID and TENCENT_SECRET_KEY in Colab secrets and re-run Step 1.')

t0 = time.time()
try:
    print(f'[QuickTest] Submitting: {QUICK_PROMPT!r} ({QUICK_FACES} faces, {QUICK_TYPE})')
    job_id = CLIENT.text_to_3d(QUICK_PROMPT, QUICK_PBR, QUICK_FACES, QUICK_TYPE, 'triangle')
    print(f'  job_id: {job_id}')
    print('[QuickTest] Waiting for completion (up to 5 min) ...')
    result = CLIENT.wait_for_completion(job_id, max_wait=300, poll=5,
                                         on_progress=lambda p, m: print(f'  [{int(p):>3d}%] {m}'))
    if not result.get('glb_url'):
        raise RuntimeError(f'No GLB: {result}')
    out = OUT_DIR / f'quicktest_{int(time.time())}.glb'
    CLIENT.download_glb(result['glb_url'], str(out))
    print(f'\n[QuickTest] DONE in {time.time() - t0:.1f}s. {out}')
    display(FileLink(str(out)))
except Exception as e:
    print(f'[QuickTest] FAILED: {e}')
    traceback.print_exc()
    raise

In [ ]:

# @title STEP 7 — Batch Generation
# @markdown Reads either a folder of images (for image-to-3D) OR a .txt file of prompts (one per line,
# @markdown for text-to-3D), and generates a .glb for each. Per-item try/except so one failure doesn't
# @markdown stop the batch. Outputs go to OUT_DIR.

import os, time, traceback, pathlib

# --- Edit these before running ---------------------------------
BATCH_MODE = 'text'  # @param ["text", "image"]
BATCH_INPUT = '/content/hy3d3_batch_prompts.txt'  # for text: a .txt file of prompts; for image: a folder of images
BATCH_FACES = 200000  # @param {type:"integer"}
BATCH_TYPE = 'LowPoly'  # @param ["Normal", "LowPoly", "Geometry", "Sketch"]
BATCH_PBR = False  # @param {type:"boolean"}
BATCH_MAX_WAIT = 600  # @param {type:"integer"}
MAX_ITEMS = 0  # 0 = no cap
# --------------------------------------------------------------

if not CLIENT.is_ready:
    raise SystemExit('Client not ready.')

if BATCH_MODE == 'text':
    list_path = pathlib.Path(BATCH_INPUT)
    if not list_path.exists():
        # Bootstrap with a few example prompts
        examples = [
            'A small red ceramic teapot, soft studio lighting, photorealistic.',
            'A cartoon dragon character, friendly, low-poly game-ready.',
            'A futuristic sci-fi helmet, scratched metal, PBR materials.',
            'A bonsai tree in a clay pot, traditional Japanese aesthetic.',
            'A vintage pocket watch, brass gears, steampunk.',
        ]
        list_path.parent.mkdir(parents=True, exist_ok=True)
        list_path.write_text('\\n'.join(examples) + '\\n')
        print(f'[Batch] No list file found, bootstrapped {len(examples)} examples into {list_path}')

    items = [l.strip() for l in list_path.read_text().splitlines() if l.strip() and not l.startswith('#')]
    if MAX_ITEMS: items = items[:MAX_ITEMS]
    print(f'[Batch] {len(items)} text prompts queued.')
    out_dir = OUT_DIR / f'batch_text_{int(time.time())}'
    out_dir.mkdir(parents=True, exist_ok=True)
    ok = 0; fail = 0
    for i, prompt in enumerate(items, 1):
        try:
            t0 = time.time()
            job_id = CLIENT.text_to_3d(prompt, BATCH_PBR, BATCH_FACES, BATCH_TYPE, 'triangle')
            result = CLIENT.wait_for_completion(job_id, max_wait=BATCH_MAX_WAIT, poll=5)
            if not result.get('glb_url'):
                print(f'  [{i:03d}] {prompt[:60]}... \\u274c no GLB in result'); fail += 1; continue
            safe = ''.join(c if c.isalnum() else '_' for c in prompt[:30]).strip('_') or f'item_{i:02d}'
            out = out_dir / f'{i:03d}_{safe}.glb'
            CLIENT.download_glb(result['glb_url'], str(out))
            print(f'  [{i:03d}] {prompt[:60]}... -> {out.name} ({time.time() - t0:.1f}s)')
            ok += 1
        except Exception as e:
            print(f'  [{i:03d}] EXCEPTION: {e}'); traceback.print_exc(); fail += 1

else:  # image mode
    folder = pathlib.Path(BATCH_INPUT)
    if not folder.is_dir():
        raise SystemExit(f'Image mode requires a folder, got: {folder}')
    exts = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
    items = sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])
    if MAX_ITEMS: items = items[:MAX_ITEMS]
    print(f'[Batch] {len(items)} images queued from {folder}')
    out_dir = OUT_DIR / f'batch_image_{int(time.time())}'
    out_dir.mkdir(parents=True, exist_ok=True)
    ok = 0; fail = 0
    for i, p in enumerate(items, 1):
        try:
            t0 = time.time()
            job_id = CLIENT.image_to_3d(str(p), BATCH_PBR, BATCH_FACES, BATCH_TYPE, 'triangle')
            result = CLIENT.wait_for_completion(job_id, max_wait=BATCH_MAX_WAIT, poll=5)
            if not result.get('glb_url'):
                print(f'  [{i:03d}] {p.name} \\u274c no GLB'); fail += 1; continue
            out = out_dir / f'{i:03d}_{p.stem}.glb'
            CLIENT.download_glb(result['glb_url'], str(out))
            print(f'  [{i:03d}] {p.name} -> {out.name} ({time.time() - t0:.1f}s)')
            ok += 1
        except Exception as e:
            print(f'  [{i:03d}] EXCEPTION: {e}'); traceback.print_exc(); fail += 1

print(f'\\n[Batch] DONE: {ok} OK / {fail} failed / {len(items)} total -> {out_dir}')
print('\\u26a0\\ufe0f  Total estimated cost: ~${0:.2f} at $0.10/job. Check your Tencent Cloud billing console for actuals.'.format(ok * 0.10))